# <span style="font-size: 44px; font-weight: bold;">Daily inventory forecasting — Colab training</span>


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, sys, subprocess, json
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith(".zip"))
root = Path("/content/inventory_project")
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        target=(root/member.filename).resolve()
        if not target.is_relative_to(root.resolve()): raise ValueError("Unsafe archive path")
    z.extractall(root)
ml = next(root.rglob("inventory-ml/train.py")).parent
subprocess.check_call([sys.executable,"-m","pip","install","-r",str(ml/"requirements.txt")])
sys.path.insert(0,str(ml))

## <span style="font-size: 26px;">Validate and inspect input</span>
The target is requested daily quantity. Stock levels and supplier receipts are excluded from demand features. Date splits keep every item on the same chronological boundary.

In [ ]:
import pandas as pd
from train import prepare
df = prepare()
display(df.head())
display(df.groupby("item_id").requested_qty.agg(["count","mean","std"]))
print("Dates:",df.date.min(),"to",df.date.max(),"; rows:",len(df))

## <span style="font-size: 26px;">Fit candidates, compare validation error, evaluate untouched test dates</span>
Selection uses daily macro SKU WAPE. The weekly baseline comparison is retained, even if the selected daily model performs worse on weekly totals.

In [ ]:
from train import main
main()
report = json.loads((ml/"artifacts/evaluation.json").read_text())
display(pd.DataFrame(report["validation_comparison"]))
print("Locked test:",report["locked_test"])
print("Baseline:",report["locked_test_baseline"])
display(pd.DataFrame(report["per_item"]))

## <span style="font-size: 26px;">Check saved inference and export</span>
This tests exact date ranges, consistent overlapping-date predictions, closures, receipt timing, units and unchanged observed records.

In [ ]:
subprocess.check_call([sys.executable,str(ml/"test_forecast.py")])
import shutil
archive_out=shutil.make_archive("inventory_daily_artifacts","zip",ml/"artifacts")
files.download(archive_out)